# Online Shoppers Regression Modeling Project

## Objective
This notebook performs feature engineering and regression modeling using the **Online Shoppers Purchasing Intention** dataset. Although the original dataset is commonly used for purchase classification, it also contains continuous engagement measures. For this regression project, the target is **`ProductRelated_Duration`**, which represents the amount of time a visitor spends on product-related pages during a session.

The notebook:

1. Loads and inspects the cleaned dataset.
2. Creates behavioral and interaction features without using target-derived information.
3. Builds and compares three regression models:
   - Multiple Linear Regression
   - Ridge Regression
   - Random Forest Regression
4. Evaluates each model using R-squared, MSE, and RMSE.
5. Uses five-fold cross-validation to assess generalization.
6. Produces visual comparisons, residual analysis, and feature-importance insights.

> **Leakage control:** `Revenue` and `PageValues` are excluded because they are post-session outcomes that could reveal information unavailable when estimating engagement duration.

In [ ]:
# Core data and numerical libraries
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn tools for preprocessing, modeling, and evaluation
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

RANDOM_STATE = 42
FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)

print("Libraries imported successfully.")

## 1. Load and Inspect the Dataset

The code searches common relative and absolute locations so that the notebook can run either from the submitted project folder or from the original working directory.

In [ ]:
# Identify the first available dataset path.
data_candidates = [
    Path("online_shoppers_cleaned.csv"),
    Path("online_shoppers_project/online_shoppers_cleaned.csv"),
    Path("/mnt/data/online_shoppers_project/online_shoppers_cleaned.csv"),
    Path("/mnt/data/online_shoppers_intention.csv"),
]

data_path = next((path for path in data_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "The dataset was not found. Place online_shoppers_cleaned.csv beside this notebook."
    )

df = pd.read_csv(data_path)

print(f"Loaded dataset from: {data_path.resolve()}")
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

In [ ]:
# Inspect data types, missing values, duplicate rows, and the regression target.
structure_summary = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Unique Values": df.nunique(dropna=False),
})

display(structure_summary)
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print()
print("Target summary (ProductRelated_Duration):")
display(df["ProductRelated_Duration"].describe().to_frame().T)

### Target Distribution

The engagement-duration target is strongly right-skewed. Most visitors spend a moderate amount of time on product pages, while a small number of sessions are extremely long. The log-transformed target is therefore used for the linear models to reduce the influence of extreme sessions.

In [ ]:
# Plot the original target distribution.
plt.figure(figsize=(9, 5))
plt.hist(df["ProductRelated_Duration"], bins=60, edgecolor="black")
plt.title("Distribution of Product-Related Duration")
plt.xlabel("Product-Related Duration")
plt.ylabel("Number of Sessions")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "01_target_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Plot log1p(target), which is less affected by extreme right-tail observations.
plt.figure(figsize=(9, 5))
plt.hist(np.log1p(df["ProductRelated_Duration"]), bins=60, edgecolor="black")
plt.title("Log-Transformed Product-Related Duration")
plt.xlabel("log(1 + Product-Related Duration)")
plt.ylabel("Number of Sessions")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "02_log_target_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

## 2. Feature Engineering

The engineered variables summarize session volume, page mix, non-product activity, and engagement quality. All calculations avoid the target itself, preventing target leakage.

Key engineered features include:

- **TotalPageCount:** total number of administrative, informational, and product pages.
- **NonProductPageCount:** pages viewed outside the product section.
- **NonProductDuration:** time spent outside product pages.
- **Average administrative/informational time:** duration per page, with zero-page sessions handled safely.
- **ProductPageShare:** percentage of viewed pages that are product-related.
- **EngagementScore:** combined inverse bounce and exit behavior.
- **BounceExitGap:** difference between exit and bounce rates.
- **SpecialDayWeekend:** interaction between special-day proximity and weekend browsing.
- **MonthNumber and Quarter:** calendar representations.
- **Log-transformed skewed predictors:** reduce the impact of large values and help linear models capture nonlinear scale relationships.

In [ ]:
def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    """Create leakage-safe behavioral features for regression modeling."""
    engineered = data.copy()

    # Standardize Boolean columns as category labels for one-hot encoding.
    engineered["Weekend"] = engineered["Weekend"].astype(str)

    # Aggregate page counts and non-product engagement.
    engineered["TotalPageCount"] = (
        engineered["Administrative"]
        + engineered["Informational"]
        + engineered["ProductRelated"]
    )
    engineered["NonProductPageCount"] = (
        engineered["Administrative"] + engineered["Informational"]
    )
    engineered["NonProductDuration"] = (
        engineered["Administrative_Duration"]
        + engineered["Informational_Duration"]
    )

    # Calculate average time per non-product page. Replace zero denominators
    # with NaN temporarily, then convert undefined averages back to zero.
    engineered["AvgAdministrativeTime"] = (
        engineered["Administrative_Duration"]
        / engineered["Administrative"].replace(0, np.nan)
    ).fillna(0)
    engineered["AvgInformationalTime"] = (
        engineered["Informational_Duration"]
        / engineered["Informational"].replace(0, np.nan)
    ).fillna(0)

    # Represent the page mix within each browsing session.
    engineered["ProductPageShare"] = (
        engineered["ProductRelated"]
        / engineered["TotalPageCount"].replace(0, np.nan)
    ).fillna(0)

    # Combine bounce and exit behavior into interpretable engagement measures.
    engineered["EngagementScore"] = (
        (1 - engineered["BounceRates"]) * (1 - engineered["ExitRates"])
    )
    engineered["BounceExitGap"] = (
        engineered["ExitRates"] - engineered["BounceRates"]
    )

    # Add a simple interaction between weekend activity and special-day proximity.
    weekend_indicator = (engineered["Weekend"] == "True").astype(int)
    engineered["SpecialDayWeekend"] = engineered["SpecialDay"] * weekend_indicator

    # Convert month labels to ordered numerical and quarterly representations.
    month_order = {
        "Feb": 2, "Mar": 3, "May": 5, "June": 6, "Jul": 7,
        "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12,
    }
    engineered["MonthNumber"] = engineered["Month"].map(month_order)
    engineered["Quarter"] = pd.cut(
        engineered["MonthNumber"],
        bins=[0, 3, 6, 9, 12],
        labels=["Q1", "Q2", "Q3", "Q4"],
        include_lowest=True,
    )

    # Apply log1p to skewed nonnegative predictors. Original variables are kept
    # so that nonlinear and linear models can choose the useful representation.
    skewed_predictors = [
        "Administrative_Duration",
        "Informational_Duration",
        "Administrative",
        "Informational",
        "ProductRelated",
        "TotalPageCount",
        "NonProductPageCount",
        "NonProductDuration",
    ]
    for column in skewed_predictors:
        engineered[f"Log_{column}"] = np.log1p(engineered[column])

    return engineered

modeling_df = engineer_features(df)
print(f"Columns before feature engineering: {df.shape[1]}")
print(f"Columns after feature engineering:  {modeling_df.shape[1]}")

display(
    modeling_df[[
        "TotalPageCount", "NonProductDuration", "ProductPageShare",
        "EngagementScore", "MonthNumber", "Quarter"
    ]].head()
)

## 3. Define Predictors and Target

`ProductRelated_Duration` is the continuous dependent variable. `Revenue` and `PageValues` are intentionally removed to prevent post-session leakage. The remaining original and engineered fields are used as predictors.

In [ ]:
TARGET = "ProductRelated_Duration"
LEAKAGE_COLUMNS = ["Revenue", "PageValues"]

y = modeling_df[TARGET].copy()
X = modeling_df.drop(columns=[TARGET] + LEAKAGE_COLUMNS)

# Create a fixed holdout set for final evaluation. The cross-validation step
# is performed only on the training portion to keep the final test set unseen.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()
numerical_features = [
    column for column in X_train.columns if column not in categorical_features
]

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows:  {len(X_test):,}")
print(f"Numerical predictors:   {len(numerical_features)}")
print(f"Categorical predictors: {len(categorical_features)}")
print()
print("Categorical fields:", categorical_features)

## 4. Preprocessing and Model Construction

The preprocessing pipeline:

- imputes numerical variables with the median,
- standardizes numerical variables for linear and Ridge regression,
- imputes categorical variables with the most common category, and
- one-hot encodes categorical variables while safely handling unseen categories.

The linear models predict `log(1 + duration)` and convert predictions back to the original scale. This approach addresses the target's strong right skew. Random Forest is trained on the original target because it naturally models nonlinear relationships and interactions.

In [ ]:
# Numerical and categorical preprocessing are combined in one reusable transformer.
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features),
])

# Base pipelines are wrapped when a log-transformed target is desired.
linear_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=0.10)),
])

forest_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=18,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

models = {
    "Multiple Linear Regression": TransformedTargetRegressor(
        regressor=linear_pipeline,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    ),
    "Ridge Regression": TransformedTargetRegressor(
        regressor=ridge_pipeline,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    ),
    "Random Forest Regression": forest_pipeline,
}

print("Models created:")
for model_name in models:
    print(f"- {model_name}")

## 5. Five-Fold Cross-Validation

Five-fold shuffled cross-validation evaluates each model repeatedly on unseen portions of the training data. The reported values are mean validation performance across folds, accompanied by standard deviations.

In [ ]:
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "R2": "r2",
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
}

cv_rows = []
for model_name, estimator in models.items():
    print(f"Cross-validating: {model_name}")
    cv_output = cross_validate(
        estimator,
        X_train,
        y_train,
        cv=cv_strategy,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    cv_rows.append({
        "Model": model_name,
        "CV R-squared Mean": cv_output["test_R2"].mean(),
        "CV R-squared SD": cv_output["test_R2"].std(),
        "CV MSE Mean": -cv_output["test_MSE"].mean(),
        "CV RMSE Mean": -cv_output["test_RMSE"].mean(),
        "CV RMSE SD": cv_output["test_RMSE"].std(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values(
    "CV RMSE Mean", ascending=True
).reset_index(drop=True)

display(cv_results.round(4))

## 6. Final Holdout Evaluation

Each model is fitted on the complete training set and evaluated once on the untouched 20% test set. Predictions below zero are clipped to zero because a session duration cannot be negative.

In [ ]:
test_rows = []
fitted_models = {}
predictions = {}

for model_name, estimator in models.items():
    estimator.fit(X_train, y_train)
    raw_predictions = estimator.predict(X_test)
    y_pred = np.maximum(raw_predictions, 0)

    fitted_models[model_name] = estimator
    predictions[model_name] = y_pred

    test_mse = mean_squared_error(y_test, y_pred)
    test_rows.append({
        "Model": model_name,
        "Test R-squared": r2_score(y_test, y_pred),
        "Test MSE": test_mse,
        "Test RMSE": np.sqrt(test_mse),
    })

holdout_results = pd.DataFrame(test_rows).sort_values(
    "Test RMSE", ascending=True
).reset_index(drop=True)

model_results = holdout_results.merge(cv_results, on="Model")
model_results = model_results[[
    "Model",
    "Test R-squared",
    "Test MSE",
    "Test RMSE",
    "CV R-squared Mean",
    "CV R-squared SD",
    "CV MSE Mean",
    "CV RMSE Mean",
    "CV RMSE SD",
]]

model_results.to_csv("model_evaluation_metrics.csv", index=False)
display(model_results.round(4))

## 7. Model-Performance Visualizations

In [ ]:
# Compare holdout and cross-validation R-squared values.
r2_plot = model_results.set_index("Model")[[
    "Test R-squared", "CV R-squared Mean"
]]

ax = r2_plot.plot(kind="bar", figsize=(10, 5))
ax.set_title("R-squared Comparison Across Regression Models")
ax.set_xlabel("Model")
ax.set_ylabel("R-squared")
ax.tick_params(axis="x", rotation=20)
ax.legend(loc="best")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "03_r2_model_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Compare holdout and cross-validation RMSE values. Lower RMSE is better.
rmse_plot = model_results.set_index("Model")[[
    "Test RMSE", "CV RMSE Mean"
]]

ax = rmse_plot.plot(kind="bar", figsize=(10, 5))
ax.set_title("RMSE Comparison Across Regression Models")
ax.set_xlabel("Model")
ax.set_ylabel("Root Mean Squared Error")
ax.tick_params(axis="x", rotation=20)
ax.legend(loc="best")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "04_rmse_model_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Select the model with the lowest holdout RMSE for detailed diagnostics.
best_model_name = model_results.sort_values("Test RMSE").iloc[0]["Model"]
best_predictions = predictions[best_model_name]

print(f"Best holdout model: {best_model_name}")

# Limit the display range to the 99th percentile so the main pattern is visible.
plot_limit = float(np.percentile(np.concatenate([y_test.values, best_predictions]), 99))

plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.35)
plt.plot([0, plot_limit], [0, plot_limit], linestyle="--")
plt.xlim(0, plot_limit)
plt.ylim(0, plot_limit)
plt.title(f"Actual vs. Predicted Duration: {best_model_name}")
plt.xlabel("Actual Product-Related Duration")
plt.ylabel("Predicted Product-Related Duration")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "05_actual_vs_predicted.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Residuals reveal whether errors change systematically across predicted values.
residuals = y_test.values - best_predictions

plt.figure(figsize=(8, 5))
plt.scatter(best_predictions, residuals, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlim(0, float(np.percentile(best_predictions, 99)))
residual_limit = float(np.percentile(np.abs(residuals), 99))
plt.ylim(-residual_limit, residual_limit)
plt.title(f"Residual Plot: {best_model_name}")
plt.xlabel("Predicted Product-Related Duration")
plt.ylabel("Residual (Actual - Predicted)")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "06_residual_plot.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Feature Influence

The following cell extracts standardized coefficients for a linear/Ridge winner or feature importances for a Random Forest winner. Larger absolute values indicate greater influence within the selected model, but they do not prove causation.

In [ ]:
best_estimator = fitted_models[best_model_name]

# Access the fitted pipeline whether the estimator is target-transformed or direct.
if isinstance(best_estimator, TransformedTargetRegressor):
    fitted_pipeline = best_estimator.regressor_
else:
    fitted_pipeline = best_estimator

fitted_preprocessor = fitted_pipeline.named_steps["preprocessor"]
fitted_regressor = fitted_pipeline.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(fitted_regressor, "coef_"):
    influence_values = np.ravel(fitted_regressor.coef_)
    influence_label = "Coefficient"
    influence_table = pd.DataFrame({
        "Feature": feature_names,
        influence_label: influence_values,
        "Absolute Influence": np.abs(influence_values),
    })
elif hasattr(fitted_regressor, "feature_importances_"):
    influence_values = fitted_regressor.feature_importances_
    influence_label = "Importance"
    influence_table = pd.DataFrame({
        "Feature": feature_names,
        influence_label: influence_values,
        "Absolute Influence": influence_values,
    })
else:
    raise AttributeError("The selected model does not expose coefficients or importances.")

# Remove technical transformer prefixes for clearer display.
influence_table["Feature"] = (
    influence_table["Feature"]
    .str.replace("numeric__", "", regex=False)
    .str.replace("categorical__", "", regex=False)
)

top_influences = influence_table.nlargest(15, "Absolute Influence").copy()
display(top_influences[["Feature", influence_label]].round(4))

plot_data = top_influences.sort_values("Absolute Influence", ascending=True)
plt.figure(figsize=(9, 6))
plt.barh(plot_data["Feature"], plot_data[influence_label])
plt.title(f"Top Feature Influences: {best_model_name}")
plt.xlabel(influence_label)
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "07_top_feature_influences.png", dpi=180, bbox_inches="tight")
plt.show()

## 9. Evaluation Summary and Modeling Insights

In [ ]:
best_row = model_results.sort_values("Test RMSE").iloc[0]
second_row = model_results.sort_values("Test RMSE").iloc[1]

print("MODEL EVALUATION SUMMARY")
print("=" * 72)
print(f"Best model: {best_row['Model']}")
print(f"Test R-squared: {best_row['Test R-squared']:.4f}")
print(f"Test MSE:       {best_row['Test MSE']:,.2f}")
print(f"Test RMSE:      {best_row['Test RMSE']:,.2f}")
print(f"CV R-squared:   {best_row['CV R-squared Mean']:.4f} "
      f"± {best_row['CV R-squared SD']:.4f}")
print(f"CV RMSE:        {best_row['CV RMSE Mean']:,.2f} "
      f"± {best_row['CV RMSE SD']:,.2f}")
print()
print(f"Second-best holdout model: {second_row['Model']}")
print(f"RMSE difference: {second_row['Test RMSE'] - best_row['Test RMSE']:,.2f}")
print()
print("KEY INSIGHTS")
print("- Product-page count and its logarithmic representation are major predictors")
print("  of how long visitors remain on product pages.")
print("- The target is strongly right-skewed; log-target modeling reduces the impact")
print("  of unusually long sessions and improves linear-model stability.")
print("- Similar holdout and cross-validation results indicate that the selected")
print("  model generalizes reasonably well rather than merely fitting one split.")
print("- Residual spread increases for very long sessions, suggesting that extreme")
print("  engagement is harder to predict and may require richer behavioral signals.")
print("- Revenue and PageValues were excluded to avoid post-session information leakage.")

## 10. Conclusion

The model with the lowest test RMSE is selected as the best-performing regression approach. Model quality is judged using both the untouched holdout set and five-fold cross-validation rather than a single training score. The analysis shows that engineered session-volume, page-mix, engagement, and log-scale features provide useful information for predicting product-page duration.

For future work, performance may improve through hyperparameter tuning, robust loss functions, gradient-boosting models, two-stage modeling for zero-duration sessions, and additional clickstream variables such as product category, device detail, referral source quality, or sequence-level navigation behavior.